# Benchmark Phase 1 v6 - Ranking direct, Walk-Forward et Optuna métier

Objectif: améliorer le scoring des marchés d'exportation pour une PME marocaine en passant d'une simple régression sur `log_return` à une approche orientée recommandation.

Cette version teste:
- validation temporelle Walk-Forward;
- régression robuste sur `log_return`;
- cible métier `market_attractiveness_target`;
- ranking direct avec LightGBM LambdaRank;
- CatBoost avec variables catégorielles;
- Optuna optimisé sur Top3, NDCG@3, Spearman et regret;
- analyse d'erreurs et importance des variables;
- figures et artefacts prêts pour le rapport PFE.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
import joblib

try:
    from lightgbm import LGBMRegressor, LGBMRanker
except Exception:
    LGBMRegressor = None
    LGBMRanker = None

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None

try:
    import optuna
except Exception:
    optuna = None

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent

DATA_DIR = ROOT / 'data' / 'raw'
FIGURES_DIR = ROOT / 'notebooks' / 'figures'
ARTIFACTS_DIR = ROOT / 'artifacts'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_OPTUNA = True
OPTUNA_TRIALS = 80
RUN_SHAP = False

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'axes.edgecolor': '#D5DAE3',
    'grid.color': '#E8ECF3',
})

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print('saved:', path)


C:\Users\HP\Desktop\MaroTrade Intelligence\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement et audit qualité des données

In [2]:
df_raw = pd.read_csv(DATA_DIR / 'ml_dataset_full.csv')
df_raw['hs_code'] = df_raw['hs_code'].astype(str).str.zfill(6)
df_raw['partner_code'] = df_raw['partner_code'].astype(str).str.replace('.0', '', regex=False).str.zfill(3)

audit = {
    'rows': int(len(df_raw)),
    'columns': int(df_raw.shape[1]),
    'years_min': int(df_raw['year'].min()),
    'years_max': int(df_raw['year'].max()),
    'duplicate_keys': int(df_raw.duplicated(['hs_code', 'year', 'partner_code']).sum()),
    'target_missing': int(df_raw['log_return'].isna().sum()),
    'target_zero_pct': float((df_raw['log_return'] == 0).mean()),
}
print(json.dumps(audit, indent=2))
display(df_raw.head())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_raw.groupby('year').size().plot(kind='bar', ax=axes[0], color='#3758F9')
axes[0].set_title('Observations par année')
df_raw['log_return'].clip(-200, 200).hist(bins=70, ax=axes[1], color='#12A594')
axes[1].set_title('Distribution cible log_return')
df_raw.isna().mean().sort_values(ascending=False).head(12).plot(kind='barh', ax=axes[2], color='#F59E0B')
axes[2].set_title('Taux de valeurs manquantes')
savefig('v6_01_data_quality_audit.png')


{
  "rows": 30193,
  "columns": 19,
  "years_min": 2015,
  "years_max": 2023,
  "duplicate_keys": 0,
  "target_missing": 0,
  "target_zero_pct": 0.0
}


,hs_code,year,partner_code,hs_desc,partner_name,value_usd,weight_kg,price_usd_kg,value_next,log_return,target_year,lag1,lag2,lag3,ma3,std3,growth_lag1,log_return_lag1,price_lag1
0,000001,2021,124,Animals; live,Canada,9638.555,0,0.0,2690.179,-127.616347,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000001,2022,124,Animals; live,Canada,2690.179,0,0.0,5114.480,64.246800,2023,9638.555,NaN,NaN,NaN,NaN,-72.089395,-127.616347,0.0
2,000001,2017,024,Animals; live,Angola,105323.997,0,0.0,108673.121,3.130320,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000001,2015,251,Animals; live,France,133744.555,0,0.0,46774.000,-105.060418,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,000001,2017,251,Animals; live,France,412055.055,0,0.0,152983.075,-99.082967,2018,46774.000,133744.555,NaN,NaN,61497.469204,780.948935,217.582948,0.0


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_01_data_quality_audit.png


## 2. Enrichissement externe et filtrage stable

On garde la logique robuste de v5.1: pas de `value_next`, pas de `price_lag1`, pas de fuite de cible.

In [3]:
ISO3_TO_UN = {
    'FRA':'250','USA':'840','DEU':'276','NLD':'528','BEL':'056','ESP':'724','ITA':'380','GBR':'826','CAN':'124','JPN':'392','CHN':'156','IND':'356','ARE':'784','SAU':'682','TUR':'792','CHE':'756','PRT':'620','SWE':'752','NOR':'579','DNK':'208','POL':'616','AUT':'040','IRL':'372','AUS':'036','BRA':'076','KOR':'410','RUS':'643','EGY':'818','DZA':'012','TUN':'788','MRT':'478','SEN':'686','CIV':'384','NGA':'566','ZAF':'710'
}
PAYS_ACCORDS_UN = set(ISO3_TO_UN.values())

DISTANCE_KM = {
    '250': 1800, '840': 6100, '276': 2500, '528': 2450, '056': 2300, '724': 1000, '380': 1900,
    '826': 2600, '124': 5900, '392': 11200, '156': 10500, '356': 8500, '784': 5800, '682': 5200,
    '792': 3900, '756': 2200, '620': 850, '752': 3400, '579': 3600, '208': 3200, '616': 3100,
    '040': 2500, '372': 2700, '036': 17800, '076': 6700, '410': 11200, '643': 5200, '818': 3600,
    '012': 1400, '788': 1600, '478': 1700, '686': 2400, '384': 3200, '566': 3000, '710': 7600,
}

def read_optional_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

accords = read_optional_csv(DATA_DIR / 'accords_maroc.csv')
wb = read_optional_csv(DATA_DIR / 'worldbank_indicators.csv')
ocde = read_optional_csv(DATA_DIR / 'ocde_risk.csv')

df = df_raw.drop(columns=[c for c in ['value_next', 'price_lag1'] if c in df_raw.columns]).copy()
funnel = [('raw', len(df))]

df = df[df['partner_code'].isin(PAYS_ACCORDS_UN)].copy()
funnel.append(('target markets only', len(df)))

df = df[df['value_usd'] >= 100_000].copy()
funnel.append(('value_usd >= 100k', len(df)))

pair_stats = df.groupby(['hs_code', 'partner_code']).agg(n_years=('year', 'nunique'), total_value=('value_usd', 'sum')).reset_index()
valid_pairs = pair_stats[(pair_stats['n_years'] >= 3) & (pair_stats['total_value'] >= 500_000)][['hs_code', 'partner_code']]
df = df.merge(valid_pairs, on=['hs_code', 'partner_code'], how='inner')
funnel.append(('pair history >=3 years and total >=500k', len(df)))

df['log_return'] = df['log_return'].clip(-100, 100)
df = df.dropna(subset=['lag1', 'lag2', 'lag3', 'log_return']).copy()
funnel.append(('lags and target available', len(df)))

if not accords.empty:
    if 'partner_code' not in accords.columns:
        code_col = 'iso3' if 'iso3' in accords.columns else ('Unnamed: 0' if 'Unnamed: 0' in accords.columns else None)
        if code_col is not None:
            accords['partner_code'] = accords[code_col].astype(str).map(ISO3_TO_UN)
    accords['partner_code'] = accords['partner_code'].astype(str).str.zfill(3)
    if 'accord_score' not in accords.columns:
        accords['accord_score'] = np.where(accords.get('type', '').astype(str).str.contains('ALE|UE|Accord', case=False, na=False), 100, 50)
    df = df.merge(accords[['partner_code', 'accord_score', 'droits']].drop_duplicates('partner_code'), on='partner_code', how='left')
else:
    df['accord_score'] = 0
    df['droits'] = 15

if not wb.empty:
    if 'partner_code' not in wb.columns:
        code_col = 'iso3' if 'iso3' in wb.columns else ('Unnamed: 0' if 'Unnamed: 0' in wb.columns else None)
        if code_col is not None:
            wb['partner_code'] = wb[code_col].astype(str).map(ISO3_TO_UN)
    wb = wb.rename(columns={'gdp_per_capita': 'wb_gdp_per_capita', 'imports_pct_gdp': 'wb_imports_pct_gdp', 'trade_pct_gdp': 'wb_trade_pct_gdp'})
    wb['partner_code'] = wb['partner_code'].astype(str).str.zfill(3)
    keep = [c for c in ['partner_code', 'wb_gdp_per_capita', 'wb_imports_pct_gdp', 'wb_trade_pct_gdp'] if c in wb.columns]
    df = df.merge(wb[keep].drop_duplicates('partner_code'), on='partner_code', how='left')
else:
    df['wb_gdp_per_capita'] = np.nan
    df['wb_imports_pct_gdp'] = np.nan
    df['wb_trade_pct_gdp'] = np.nan

if not ocde.empty:
    if 'partner_code' not in ocde.columns:
        code_col = 'iso3' if 'iso3' in ocde.columns else ('Unnamed: 0' if 'Unnamed: 0' in ocde.columns else None)
        if code_col is not None:
            ocde['partner_code'] = ocde[code_col].astype(str).map(ISO3_TO_UN)
    if 'ocde_risk_score' not in ocde.columns:
        ocde['ocde_risk_score'] = ocde['category'] if 'category' in ocde.columns else (100 - ocde['score']) / 20 if 'score' in ocde.columns else np.nan
    ocde['partner_code'] = ocde['partner_code'].astype(str).str.zfill(3)
    df = df.merge(ocde[['partner_code', 'ocde_risk_score']].drop_duplicates('partner_code'), on='partner_code', how='left')
else:
    df['ocde_risk_score'] = np.nan

df['distance_km'] = df['partner_code'].map(DISTANCE_KM).astype(float)
df['wb_available'] = (~df['wb_gdp_per_capita'].isna()).astype(int)
df['trend_score'] = df.get('trend_score', 50)

funnel_df = pd.DataFrame(funnel, columns=['step', 'rows'])
display(funnel_df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(funnel_df['step'], funnel_df['rows'], color='#3758F9')
ax.invert_yaxis()
ax.set_title('Funnel de filtrage v6')
ax.set_xlabel('Nombre de lignes')
savefig('v6_02_filter_funnel.png')


,step,rows
0,raw,30193
1,target markets only,12682
2,value_usd >= 100k,8173
3,pair history >=3 years and total >=500k,7699
4,lags and target available,5125


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_02_filter_funnel.png


## 3. Feature engineering v6

On compare trois familles: noyau v5.1 stable, features avancées, et variables catégorielles pour CatBoost.

In [4]:
df_fe = df.sort_values(['hs_code', 'partner_code', 'year']).copy()
g = df_fe.groupby(['hs_code', 'partner_code'], group_keys=False)

if 'market_share' not in df_fe.columns:
    market_total = df_fe.groupby(['hs_code', 'year'])['value_usd'].transform('sum').replace(0, np.nan)
    df_fe['market_share'] = (df_fe['value_usd'] / market_total).fillna(0)
if 'cagr_3y' not in df_fe.columns:
    df_fe['cagr_3y'] = ((df_fe['value_usd'] / df_fe['lag3'].replace(0, np.nan)) ** (1 / 3) - 1).replace([np.inf, -np.inf], np.nan).clip(-5, 5)
if 'trend_score' not in df_fe.columns:
    df_fe['trend_score'] = 50

df_fe['log_value_usd'] = np.log1p(df_fe['value_usd'])
df_fe['log_lag1'] = np.log1p(df_fe['lag1'])
df_fe['log_ma3'] = np.log1p(df_fe['ma3'])
df_fe['price_usd_kg'] = df_fe['price_usd_kg'].replace([np.inf, -np.inf], np.nan)
df_fe['log_price_usd_kg'] = np.log1p(df_fe['price_usd_kg'].clip(lower=0))
df_fe['log_weight_kg'] = np.log1p(df_fe['weight_kg'].clip(lower=0))
df_fe['log_distance'] = np.log1p(df_fe['distance_km'])
df_fe['market_share_lag1'] = g['market_share'].shift(1)
df_fe['price_growth_lag1'] = g['price_usd_kg'].pct_change().shift(1).replace([np.inf, -np.inf], np.nan).clip(-5, 5)
df_fe['value_growth_2y'] = (df_fe['lag1'] / df_fe['lag3'].replace(0, np.nan) - 1).replace([np.inf, -np.inf], np.nan).clip(-5, 5)
df_fe['country_hs_rank_value'] = df_fe.groupby(['year', 'hs_code'])['value_usd'].rank(pct=True)
df_fe['country_hs_rank_growth'] = df_fe.groupby(['year', 'hs_code'])['growth_lag1'].rank(pct=True)
df_fe['access_score'] = (
    0.35 * df_fe['accord_score'].fillna(0)
    + 0.25 * (100 - df_fe['droits'].fillna(df_fe['droits'].median()).clip(0, 100))
    + 0.20 * (100 - df_fe['ocde_risk_score'].fillna(df_fe['ocde_risk_score'].median()).clip(0, 7) * 100 / 7)
    + 0.20 * (100 - df_fe['distance_km'].rank(pct=True).fillna(0.5) * 100)
)
df_fe['risk_adjusted_growth'] = df_fe['growth_lag1'].fillna(0) - df_fe['ocde_risk_score'].fillna(df_fe['ocde_risk_score'].median()) * 2
df_fe['demand_openness'] = df_fe['wb_imports_pct_gdp'].fillna(df_fe['wb_imports_pct_gdp'].median()) + 0.5 * df_fe.get('wb_trade_pct_gdp', 0).fillna(0)

# Historical encodings, shifted to avoid leakage.
df_fe['hs_mean_return_lag'] = df_fe.groupby('hs_code')['log_return'].transform(lambda s: s.expanding().mean().shift(1))
df_fe['country_mean_return_lag'] = df_fe.groupby('partner_code')['log_return'].transform(lambda s: s.expanding().mean().shift(1))
df_fe['route_mean_return_lag'] = g['log_return'].transform(lambda s: s.expanding().mean().shift(1))

CORE_FEATURES = [
    'log_value_usd', 'log_lag1', 'log_ma3', 'std3', 'growth_lag1', 'log_return_lag1',
    'price_usd_kg', 'market_share', 'cagr_3y', 'accord_score', 'droits',
    'wb_gdp_per_capita', 'wb_imports_pct_gdp', 'wb_available', 'ocde_risk_score',
    'distance_km', 'trend_score'
]
ADVANCED_FEATURES = [
    'log_weight_kg', 'log_price_usd_kg', 'market_share_lag1', 'price_growth_lag1',
    'value_growth_2y', 'country_hs_rank_value', 'country_hs_rank_growth', 'access_score',
    'risk_adjusted_growth', 'demand_openness', 'log_distance', 'hs_mean_return_lag',
    'country_mean_return_lag', 'route_mean_return_lag'
]
FEATURE_SETS = {
    'core_v5_1_stable': CORE_FEATURES,
    'core_plus_advanced_v6': CORE_FEATURES + ADVANCED_FEATURES,
}
CAT_FEATURES = ['hs_code', 'partner_code']

for col in set(CORE_FEATURES + ADVANCED_FEATURES):
    if col not in df_fe.columns:
        df_fe[col] = np.nan
    df_fe[col] = pd.to_numeric(df_fe[col], errors='coerce').replace([np.inf, -np.inf], np.nan)

print('df_fe:', df_fe.shape)
print('features core:', len(CORE_FEATURES), 'advanced:', len(ADVANCED_FEATURES))
display(df_fe[CORE_FEATURES + ADVANCED_FEATURES].isna().mean().sort_values(ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df_fe[CORE_FEATURES + ADVANCED_FEATURES].isna().mean().sort_values(ascending=False).head(15).plot(kind='barh', ax=axes[0], color='#F59E0B')
axes[0].set_title('Missing rate features v6')
df_fe[CORE_FEATURES].corr(numeric_only=True).abs().mean().sort_values(ascending=False).head(12).plot(kind='barh', ax=axes[1], color='#7C3AED')
axes[1].set_title('Corrélation moyenne absolue - core')
savefig('v6_03_feature_diagnostics.png')


df_fe: (5125, 45)
features core: 17 advanced: 14


price_growth_lag1          1.000000
accord_score               0.473951
wb_imports_pct_gdp         0.473951
wb_gdp_per_capita          0.473951
droits                     0.473951
ocde_risk_score            0.473951
market_share_lag1          0.201171
route_mean_return_lag      0.201171
hs_mean_return_lag         0.015805
country_mean_return_lag    0.006049
growth_lag1                0.000000
price_usd_kg               0.000000
log_return_lag1            0.000000
log_lag1                   0.000000
log_value_usd              0.000000
dtype: float64

saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_03_feature_diagnostics.png


## 4. Cibles: log_return robuste et market_attractiveness_target

In [5]:
df_fe['target_log_return_winsor'] = df_fe['log_return'].clip(df_fe['log_return'].quantile(0.02), df_fe['log_return'].quantile(0.98))
df_fe['market_size_score'] = df_fe.groupby('year')['value_usd'].rank(pct=True).fillna(0) * 100
df_fe['stability_score'] = 100 - df_fe['std3'].rank(pct=True).fillna(0.5) * 100
df_fe['accessibility_score'] = df_fe['access_score'].fillna(df_fe['access_score'].median())
df_fe['future_growth_score'] = df_fe['target_log_return_winsor'].clip(-100, 100)
df_fe['market_attractiveness_target'] = (
    0.50 * df_fe['future_growth_score']
    + 0.25 * df_fe['stability_score']
    + 0.15 * df_fe['market_size_score']
    + 0.10 * df_fe['accessibility_score']
)

TARGETS = {
    'log_return': 'log_return',
    'log_return_winsor': 'target_log_return_winsor',
    'market_attractiveness': 'market_attractiveness_target',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, col) in zip(axes, TARGETS.items()):
    df_fe[col].hist(bins=60, ax=ax, color='#3758F9', alpha=0.85)
    ax.set_title(name)
savefig('v6_04_target_comparison.png')


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_04_target_comparison.png


## 5. Métriques métier et Walk-Forward

In [6]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)

def safe_spearman(y_true, y_pred):
    if len(np.unique(y_pred)) < 2 or len(np.unique(y_true)) < 2:
        return 0.0
    val = spearmanr(y_true, y_pred).correlation
    return float(0.0 if pd.isna(val) else val)

def ranking_metrics(y_true, y_pred, hs_codes, years, k_values=(1, 3, 5)):
    frame = pd.DataFrame({'y': y_true, 'pred': y_pred, 'hs_code': hs_codes, 'year': years})
    groups = []
    for _, gdf in frame.groupby(['hs_code', 'year']):
        if len(gdf) < 3:
            continue
        gdf = gdf.copy()
        best_actual_idx = gdf['y'].idxmax()
        pred_ranked = gdf.sort_values('pred', ascending=False)
        actual_ranked = gdf.sort_values('y', ascending=False)
        row = {'regret': float(actual_ranked.iloc[0]['y'] - pred_ranked.iloc[0]['y'])}
        for k in k_values:
            top_pred = set(pred_ranked.head(k).index)
            row[f'top{k}'] = float(best_actual_idx in top_pred)
            gains = pred_ranked.head(k)['y'].clip(lower=0).values
            ideal = actual_ranked.head(k)['y'].clip(lower=0).values
            denom = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
            dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains))
            row[f'ndcg@{k}'] = float(dcg / denom) if denom > 0 else 0.0
        groups.append(row)
    if not groups:
        return {'groups': 0, 'top1': 0, 'top3': 0, 'top5': 0, 'ndcg@3': 0, 'mean_regret': 0}
    out = pd.DataFrame(groups).mean(numeric_only=True).to_dict()
    out['groups'] = len(groups)
    out['mean_regret'] = float(pd.DataFrame(groups)['regret'].mean())
    return out

def eval_all(y_true, y_pred, hs_codes, years):
    r = ranking_metrics(y_true, y_pred, hs_codes, years)
    return {
        'rmse': rmse(y_true, y_pred),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'spearman': safe_spearman(y_true, y_pred),
        **r,
    }

def business_score(m):
    regret_norm = min(max(m.get('mean_regret', 0) / 100, 0), 3)
    return float(0.35 * m.get('top3', 0) + 0.30 * m.get('ndcg@3', 0) + 0.20 * m.get('spearman', 0) - 0.15 * regret_norm)

def make_walk_forward_splits(data, min_train_years=3):
    years = sorted(data['year'].unique())
    splits = []
    for test_year in years:
        train_years = [y for y in years if y < test_year]
        if len(train_years) < min_train_years:
            continue
        tr = data[data['year'].isin(train_years)].copy()
        te = data[data['year'] == test_year].copy()
        if len(tr) and len(te):
            splits.append((test_year, tr, te))
    return splits

latest_feature_year = int(df_fe['year'].max())
val_years = [latest_feature_year - 2, latest_feature_year - 1]
train = df_fe[df_fe['year'] < val_years[0]].copy()
val = df_fe[df_fe['year'].isin(val_years)].copy()
test = df_fe[df_fe['year'] == latest_feature_year].copy()
wf_splits = make_walk_forward_splits(df_fe)

print('holdout split:', len(train), len(val), len(test), 'test target year:', latest_feature_year + 1)
print('walk-forward folds:', [(y, len(tr), len(te)) for y, tr, te in wf_splits])


holdout split: 2572 1733 820 test target year: 2024
walk-forward folds: [(np.int64(2021), 2572, 865), (np.int64(2022), 3437, 868), (np.int64(2023), 4305, 820)]


## 6. Préprocessing et modèles de régression

In [7]:
def make_xy(train_df, eval_df, features, target):
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
    ])
    X_train = pipe.fit_transform(train_df[features])
    X_eval = pipe.transform(eval_df[features])
    y_train = train_df[target].values.astype(float)
    y_eval = eval_df[target].values.astype(float)
    return X_train, y_train, X_eval, y_eval, pipe

def regression_factories():
    models = {
        'Ridge': lambda: Ridge(alpha=30.0),
        'Huber': lambda: HuberRegressor(alpha=0.001, epsilon=1.35, max_iter=500),
        'ExtraTrees': lambda: ExtraTreesRegressor(n_estimators=400, max_depth=10, min_samples_leaf=8, random_state=RANDOM_STATE, n_jobs=-1),
        'RandomForest': lambda: RandomForestRegressor(n_estimators=350, max_depth=9, min_samples_leaf=8, random_state=RANDOM_STATE, n_jobs=-1),
    }
    if LGBMRegressor is not None:
        models['LightGBM'] = lambda: LGBMRegressor(n_estimators=450, max_depth=4, learning_rate=0.035, num_leaves=12, min_child_samples=25, reg_alpha=1.0, reg_lambda=8.0, subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, verbose=-1)
    if XGBRegressor is not None:
        models['XGBoost'] = lambda: XGBRegressor(n_estimators=450, max_depth=4, learning_rate=0.035, subsample=0.85, colsample_bytree=0.85, min_child_weight=8, reg_alpha=1.0, reg_lambda=8.0, random_state=RANDOM_STATE, verbosity=0)
    return models

RESULTS = []
FITTED = {}

for target_name, target_col in TARGETS.items():
    for feature_set, features in FEATURE_SETS.items():
        X_tr, y_tr, X_val, y_val, pipe = make_xy(train, val, features, target_col)
        X_te = pipe.transform(test[features])
        y_te = test[target_col].values.astype(float)
        for model_name, factory in regression_factories().items():
            model = factory()
            model.fit(X_tr, y_tr)
            pv = model.predict(X_val)
            pt = model.predict(X_te)
            mv = eval_all(y_val, pv, val['hs_code'].values, val['year'].values)
            mt = eval_all(y_te, pt, test['hs_code'].values, test['year'].values)
            key = f'{model_name}__{target_name}__{feature_set}'
            RESULTS.append({'key': key, 'model_type': 'regression', 'model': model_name, 'target': target_name, 'feature_set': feature_set, 'business_score': business_score(mv), **{f'val_{k}': v for k, v in mv.items()}, **{f'test_{k}': v for k, v in mt.items()}})
            FITTED[key] = {'model': model, 'preprocessor': pipe, 'features': features, 'target': target_col, 'pred_test': pt}
            print(key, 'val_top3', round(mv.get('top3', 0), 3), 'test_top3', round(mt.get('top3', 0), 3), 'test_spearman', round(mt.get('spearman', 0), 3))


Ridge__log_return__core_v5_1_stable val_top3 0.372 test_top3 0.382 test_spearman -0.017


Huber__log_return__core_v5_1_stable val_top3 0.416 test_top3 0.353 test_spearman -0.024


ExtraTrees__log_return__core_v5_1_stable val_top3 0.387 test_top3 0.412 test_spearman -0.014


RandomForest__log_return__core_v5_1_stable val_top3 0.431 test_top3 0.382 test_spearman 0.026


LightGBM__log_return__core_v5_1_stable val_top3 0.445 test_top3 0.324 test_spearman 0.033


...

Huber__market_attractiveness__core_plus_advanced_v6 val_top3 0.453 test_top3 0.544 test_spearman 0.167


ExtraTrees__market_attractiveness__core_plus_advanced_v6 val_top3 0.482 test_top3 0.485 test_spearman 0.202


RandomForest__market_attractiveness__core_plus_advanced_v6 val_top3 0.46 test_top3 0.559 test_spearman 0.237


LightGBM__market_attractiveness__core_plus_advanced_v6 val_top3 0.474 test_top3 0.588 test_spearman 0.184


XGBoost__market_attractiveness__core_plus_advanced_v6 val_top3 0.445 test_top3 0.544 test_spearman 0.185


## 7. Ranking direct avec LightGBM LambdaRank

In [8]:
def sorted_group_sizes(data):
    return data.groupby(['hs_code', 'year']).size().values

def relevance_labels(data, target_col, max_grade=30):
    # LambdaRank works best with non-negative relevance grades.
    rel = data.groupby(['hs_code', 'year'])[target_col].rank(pct=True, method='average')
    rel = (rel.fillna(0.0) * max_grade).round().astype(int).clip(0, max_grade)
    return rel.values

def fit_lgbm_ranker(train_df, val_df, test_df, features, target_col, params=None):
    if LGBMRanker is None:
        return None
    params = params or {}
    base_params = dict(
        objective='lambdarank',
        metric='ndcg',
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=20,
        reg_alpha=1.0,
        reg_lambda=6.0,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        verbose=-1,
    )
    base_params.update(params)
    tr = train_df.sort_values(['hs_code', 'year']).copy()
    va = val_df.sort_values(['hs_code', 'year']).copy()
    te = test_df.sort_values(['hs_code', 'year']).copy()
    imputer = SimpleImputer(strategy='median')
    X_tr = imputer.fit_transform(tr[features])
    X_va = imputer.transform(va[features])
    X_te = imputer.transform(te[features])
    ranker = LGBMRanker(**base_params)
    y_tr_rank = relevance_labels(tr, target_col)
    y_va_rank = relevance_labels(va, target_col)
    ranker.fit(
        X_tr,
        y_tr_rank,
        group=sorted_group_sizes(tr),
        eval_set=[(X_va, y_va_rank)],
        eval_group=[sorted_group_sizes(va)],
        eval_at=[3, 5],
    )
    return ranker, imputer, tr, va, te, ranker.predict(X_va), ranker.predict(X_te)

if LGBMRanker is not None:
    for target_name, target_col in TARGETS.items():
        for feature_set, features in FEATURE_SETS.items():
            fitted = fit_lgbm_ranker(train, val, test, features, target_col)
            if fitted is None:
                continue
            model, imputer, tr_s, va_s, te_s, pv, pt = fitted
            mv = eval_all(va_s[target_col].values, pv, va_s['hs_code'].values, va_s['year'].values)
            mt = eval_all(te_s[target_col].values, pt, te_s['hs_code'].values, te_s['year'].values)
            key = f'LGBMRanker__{target_name}__{feature_set}'
            RESULTS.append({'key': key, 'model_type': 'ranker', 'model': 'LGBMRanker', 'target': target_name, 'feature_set': feature_set, 'business_score': business_score(mv), **{f'val_{k}': v for k, v in mv.items()}, **{f'test_{k}': v for k, v in mt.items()}})
            FITTED[key] = {'model': model, 'preprocessor': imputer, 'features': features, 'target': target_col, 'pred_test': pt, 'test_sorted': te_s}
            print(key, 'val_top3', round(mv.get('top3', 0), 3), 'test_top3', round(mt.get('top3', 0), 3), 'test_ndcg3', round(mt.get('ndcg@3', 0), 3))
else:
    print('LightGBM Ranker indisponible')


LGBMRanker__log_return__core_v5_1_stable val_top3 0.409 test_top3 0.338 test_ndcg3 0.717


LGBMRanker__log_return__core_plus_advanced_v6 val_top3 0.431 test_top3 0.353 test_ndcg3 0.745


LGBMRanker__log_return_winsor__core_v5_1_stable val_top3 0.409 test_top3 0.338 test_ndcg3 0.717


LGBMRanker__log_return_winsor__core_plus_advanced_v6 val_top3 0.431 test_top3 0.353 test_ndcg3 0.745


LGBMRanker__market_attractiveness__core_v5_1_stable val_top3 0.423 test_top3 0.485 test_ndcg3 0.769


LGBMRanker__market_attractiveness__core_plus_advanced_v6 val_top3 0.496 test_top3 0.456 test_ndcg3 0.774


## 8. CatBoost avec variables catégorielles

In [9]:
if CatBoostRegressor is not None:
    for target_name, target_col in TARGETS.items():
        features = FEATURE_SETS['core_plus_advanced_v6'] + CAT_FEATURES
        cols = list(dict.fromkeys(features + [target_col, 'year']))
        tr = train[cols].copy()
        va = val[cols].copy()
        te = test[cols].copy()
        for c in CAT_FEATURES:
            tr[c] = tr[c].astype(str)
            va[c] = va[c].astype(str)
            te[c] = te[c].astype(str)
        model = CatBoostRegressor(
            iterations=700,
            depth=5,
            learning_rate=0.035,
            loss_function='RMSE',
            eval_metric='RMSE',
            l2_leaf_reg=8.0,
            min_data_in_leaf=20,
            random_seed=RANDOM_STATE,
            verbose=0,
            allow_writing_files=False,
        )
        model.fit(tr[features], tr[target_col], cat_features=CAT_FEATURES, eval_set=(va[features], va[target_col]))
        pv = model.predict(va[features])
        pt = model.predict(te[features])
        mv = eval_all(va[target_col].values, pv, va['hs_code'].values, va['year'].values)
        mt = eval_all(te[target_col].values, pt, te['hs_code'].values, te['year'].values)
        key = f'CatBoostCategorical__{target_name}__core_plus_advanced_v6'
        RESULTS.append({'key': key, 'model_type': 'categorical_regression', 'model': 'CatBoostCategorical', 'target': target_name, 'feature_set': 'core_plus_advanced_v6+categorical', 'business_score': business_score(mv), **{f'val_{k}': v for k, v in mv.items()}, **{f'test_{k}': v for k, v in mt.items()}})
        FITTED[key] = {'model': model, 'preprocessor': None, 'features': features, 'target': target_col, 'pred_test': pt}
        print(key, 'val_top3', round(mv.get('top3', 0), 3), 'test_top3', round(mt.get('top3', 0), 3))
else:
    print('CatBoost indisponible')


CatBoostCategorical__log_return__core_plus_advanced_v6 val_top3 0.416 test_top3 0.397


CatBoostCategorical__log_return_winsor__core_plus_advanced_v6 val_top3 0.416 test_top3 0.397


CatBoostCategorical__market_attractiveness__core_plus_advanced_v6 val_top3 0.431 test_top3 0.588


## 9. Optuna métier sur LightGBM Ranker

L'objectif Optuna optimise directement la métrique qui ressemble au besoin utilisateur: Top3 + NDCG@3 + Spearman - regret.

In [10]:
OPTUNA_RESULTS = {}

if RUN_OPTUNA and optuna is not None and LGBMRanker is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    opt_target_name = 'market_attractiveness'
    opt_target_col = TARGETS[opt_target_name]
    opt_features = FEATURE_SETS['core_plus_advanced_v6']

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 250, 900),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08),
            'num_leaves': trial.suggest_int('num_leaves', 6, 40),
            'max_depth': trial.suggest_int('max_depth', 2, 8),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 25.0),
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
        }
        fitted = fit_lgbm_ranker(train, val, test, opt_features, opt_target_col, params=params)
        _, _, _, va_s, _, pv, _ = fitted
        mv = eval_all(va_s[opt_target_col].values, pv, va_s['hs_code'].values, va_s['year'].values)
        return business_score(mv)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    best_params = study.best_params
    fitted = fit_lgbm_ranker(train, val, test, opt_features, opt_target_col, params=best_params)
    model, imputer, tr_s, va_s, te_s, pv, pt = fitted
    mv = eval_all(va_s[opt_target_col].values, pv, va_s['hs_code'].values, va_s['year'].values)
    mt = eval_all(te_s[opt_target_col].values, pt, te_s['hs_code'].values, te_s['year'].values)
    key = 'LGBMRankerOptuna__market_attractiveness__core_plus_advanced_v6'
    RESULTS.append({'key': key, 'model_type': 'ranker_optuna', 'model': 'LGBMRankerOptuna', 'target': opt_target_name, 'feature_set': 'core_plus_advanced_v6', 'business_score': business_score(mv), **{f'val_{k}': v for k, v in mv.items()}, **{f'test_{k}': v for k, v in mt.items()}})
    FITTED[key] = {'model': model, 'preprocessor': imputer, 'features': opt_features, 'target': opt_target_col, 'pred_test': pt, 'test_sorted': te_s}
    OPTUNA_RESULTS[key] = {'best_value': float(study.best_value), 'params': best_params, 'trials': OPTUNA_TRIALS}
    print('Optuna best:', study.best_value, best_params)
else:
    print('Optuna ranking non exécuté:', {'RUN_OPTUNA': RUN_OPTUNA, 'optuna': optuna is not None, 'LGBMRanker': LGBMRanker is not None})


  0%|          | 0/80 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.302655:   0%|          | 0/80 [00:03<?, ?it/s]

Best trial: 0. Best value: 0.302655:   1%|▏         | 1/80 [00:03<04:11,  3.18s/it]

Best trial: 0. Best value: 0.302655:   1%|▏         | 1/80 [00:06<04:11,  3.18s/it]

Best trial: 0. Best value: 0.302655:   2%|▎         | 2/80 [00:06<04:34,  3.52s/it]

...

Best trial: 11. Best value: 0.331659:  99%|█████████▉| 79/80 [03:58<00:03,  3.20s/it]

Best trial: 11. Best value: 0.331659:  99%|█████████▉| 79/80 [04:02<00:03,  3.20s/it]

Best trial: 11. Best value: 0.331659: 100%|██████████| 80/80 [04:02<00:00,  3.57s/it]

Best trial: 11. Best value: 0.331659: 100%|██████████| 80/80 [04:02<00:00,  3.04s/it]

Optuna best: 0.3316591475487386 {'n_estimators': 654, 'learning_rate': 0.026481315970305303, 'num_leaves': 25, 'max_depth': 2, 'min_child_samples': 29, 'reg_alpha': 1.8150633831294023, 'reg_lambda': 24.236524358408847, 'subsample': 0.903096730175725, 'colsample_bytree': 0.892082838181189}


## 10. Evaluation Walk-Forward du meilleur candidat

In [11]:
summary = pd.DataFrame(RESULTS).sort_values(['business_score', 'val_top3', 'val_ndcg@3'], ascending=False)
display(summary[['key', 'model', 'target', 'feature_set', 'business_score', 'val_top3', 'val_ndcg@3', 'test_top3', 'test_ndcg@3', 'test_mean_regret', 'test_rmse', 'test_mae', 'test_spearman', 'test_r2']].head(20))

BEST_KEY = summary.iloc[0]['key']
BEST = FITTED[BEST_KEY]
print('BEST_KEY =', BEST_KEY)

def fit_eval_for_fold(model_name, target_col, features, tr, te):
    # Walk-forward evaluation uses a stable LightGBM ranker if selected model is ranker-like, otherwise LightGBM regressor.
    if 'Ranker' in model_name and LGBMRanker is not None:
        cut = int(tr['year'].max())
        inner_train = tr[tr['year'] < cut].copy()
        inner_val = tr[tr['year'] == cut].copy()
        if len(inner_train) < 50 or len(inner_val) < 20:
            inner_train = tr.copy()
            inner_val = tr.copy()
        fitted = fit_lgbm_ranker(inner_train, inner_val, te, features, target_col)
        _, _, _, _, te_s, _, pt = fitted
        return eval_all(te_s[target_col].values, pt, te_s['hs_code'].values, te_s['year'].values)
    else:
        model = LGBMRegressor(n_estimators=350, max_depth=4, learning_rate=0.04, num_leaves=12, min_child_samples=25, reg_alpha=1, reg_lambda=8, random_state=RANDOM_STATE, verbose=-1) if LGBMRegressor is not None else Ridge(alpha=30)
        X_tr, y_tr, X_te, y_te, pipe = make_xy(tr, te, features, target_col)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        return eval_all(y_te, pred, te['hs_code'].values, te['year'].values)

wf_rows = []
for year, tr, te in wf_splits:
    m = fit_eval_for_fold(BEST_KEY, BEST['target'], BEST['features'], tr, te)
    wf_rows.append({'year': int(year), 'business_score': business_score(m), **m})
wf_df = pd.DataFrame(wf_rows)
display(wf_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
wf_df.plot(x='year', y='top3', marker='o', ax=axes[0], color='#3758F9', legend=False)
axes[0].set_title('Walk-Forward Top3')
wf_df.plot(x='year', y='ndcg@3', marker='o', ax=axes[1], color='#12A594', legend=False)
axes[1].set_title('Walk-Forward NDCG@3')
wf_df.plot(x='year', y='mean_regret', marker='o', ax=axes[2], color='#EF4444', legend=False)
axes[2].set_title('Walk-Forward regret')
for ax in axes: ax.grid(True)
savefig('v6_05_walk_forward_best_model.png')


,key,model,target,feature_set,business_score,val_top3,val_ndcg@3,test_top3,test_ndcg@3,test_mean_regret,test_rmse,test_mae,test_spearman,test_r2
26,ExtraTrees__market_attractiveness__core_v5_1_s...,ExtraTrees,market_attractiveness,core_v5_1_stable,0.350588,0.489051,0.635317,0.500000,0.809661,19.167714,41.323377,39.029989,0.216342,-1.660266
32,ExtraTrees__market_attractiveness__core_plus_a...,ExtraTrees,market_attractiveness,core_plus_advanced_v6,0.338097,0.481752,0.620058,0.485294,0.816499,20.436810,40.953667,38.619442,0.201777,-1.612877
30,Ridge__market_attractiveness__core_plus_advanc...,Ridge,market_attractiveness,core_plus_advanced_v6,0.334397,0.459854,0.629285,0.573529,0.808972,21.971694,41.970470,39.461959,0.221902,-1.744233
28,LightGBM__market_attractiveness__core_v5_1_stable,LightGBM,market_attractiveness,core_v5_1_stable,0.334270,0.459854,0.610817,0.529412,0.801978,21.152593,41.870594,39.180295,0.187297,-1.731188
45,LGBMRankerOptuna__market_attractiveness__core_...,LGBMRankerOptuna,market_attractiveness,core_plus_advanced_v6,0.331659,0.518248,0.619741,0.485294,0.773224,24.316446,66.647262,63.697279,-0.006881,-5.919871
31,Huber__market_attractiveness__core_plus_advanc...,Huber,market_attractiveness,core_plus_advanced_v6,0.330949,0.452555,0.632924,0.544118,0.806228,19.598497,41.379445,38.795837,0.167355,-1.667489
34,LightGBM__market_attractiveness__core_plus_adv...,LightGBM,market_attractiveness,core_plus_advanced_v6,0.326783,0.474453,0.606142,0.588235,0.798121,24.351088,40.732237,38.127403,0.183565,-1.584699
33,RandomForest__market_attractiveness__core_plus...,RandomForest,market_attractiveness,core_plus_advanced_v6,0.325937,0.459854,0.613154,0.558824,0.811320,20.737477,40.858311,38.561785,0.236964,-1.600724
29,XGBoost__market_attractiveness__core_v5_1_stable,XGBoost,market_attractiveness,core_v5_1_stable,0.322870,0.430657,0.608279,0.558824,0.805361,21.686069,41.912889,39.170384,0.191204,-1.736709
35,XGBoost__market_attractiveness__core_plus_adva...,XGBoost,market_attractiveness,core_plus_advanced_v6,0.321765,0.445255,0.613395,0.544118,0.795095,24.521920,40.687587,38.012112,0.185200,-1.579035


BEST_KEY = ExtraTrees__market_attractiveness__core_v5_1_stable


,year,business_score,rmse,mae,r2,spearman,regret,top1,ndcg@1,top3,ndcg@3,top5,ndcg@5,groups,mean_regret
0,2021,0.324553,26.865556,21.288769,0.026713,0.198322,31.230164,0.217391,0.565830,0.434783,0.598532,0.594203,0.662362,69,31.230164
1,2022,0.304104,26.629409,20.627898,0.014164,0.162289,33.132181,0.132353,0.497363,0.411765,0.590756,0.647059,0.648882,68,33.132181
2,2023,0.419166,42.187232,39.740972,-1.772653,0.237545,26.635104,0.088235,0.687503,0.500000,0.788700,0.691176,0.827265,68,26.635104


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_05_walk_forward_best_model.png


## 11. Analyse d'erreurs, importance des variables et figures finales

In [12]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
top = summary.head(12)
axes[0, 0].barh(top['key'], top['business_score'], color='#3758F9')
axes[0, 0].invert_yaxis(); axes[0, 0].set_title('Top modèles par score métier validation')
axes[0, 1].scatter(summary['test_rmse'], summary['test_spearman'], s=90, c=summary['business_score'], cmap='viridis')
axes[0, 1].set_xlabel('Test RMSE'); axes[0, 1].set_ylabel('Test Spearman'); axes[0, 1].set_title('Compromis RMSE / Spearman')
axes[1, 0].scatter(summary['test_mean_regret'], summary['test_ndcg@3'], s=90, c=summary['val_top3'], cmap='plasma')
axes[1, 0].set_xlabel('Test regret'); axes[1, 0].set_ylabel('Test NDCG@3'); axes[1, 0].set_title('Qualité ranking vs regret')
target_counts = summary.groupby('target')['business_score'].max().sort_values(ascending=False)
target_counts.plot(kind='bar', ax=axes[1, 1], color='#12A594')
axes[1, 1].set_title('Meilleur score métier par cible')
savefig('v6_06_final_model_comparison.png')

# Error analysis on test for best model.
pred_test = np.array(BEST['pred_test'])
test_for_error = BEST.get('test_sorted', test).copy()
target_col = BEST['target']
test_for_error['y_pred'] = pred_test
test_for_error['abs_error'] = (test_for_error[target_col] - test_for_error['y_pred']).abs()

err_hs = test_for_error.groupby('hs_code').agg(n=('abs_error', 'size'), mae=('abs_error', 'mean'), mean_target=(target_col, 'mean')).query('n >= 3').sort_values('mae', ascending=False).head(15)
err_country = test_for_error.groupby('partner_code').agg(n=('abs_error', 'size'), mae=('abs_error', 'mean'), mean_target=(target_col, 'mean')).query('n >= 3').sort_values('mae', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
err_hs['mae'].plot(kind='barh', ax=axes[0], color='#EF4444')
axes[0].invert_yaxis(); axes[0].set_title('Plus fortes erreurs par HS code')
err_country['mae'].plot(kind='barh', ax=axes[1], color='#F59E0B')
axes[1].invert_yaxis(); axes[1].set_title('Plus fortes erreurs par pays')
savefig('v6_07_error_analysis.png')

# Importance.
importance = None
model = BEST['model']
features = BEST['features']
if hasattr(model, 'feature_importances_'):
    importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
elif hasattr(model, 'coef_'):
    importance = pd.Series(np.abs(model.coef_), index=features).sort_values(ascending=False)

if importance is not None:
    fig, ax = plt.subplots(figsize=(8, 7))
    importance.head(20).sort_values().plot(kind='barh', ax=ax, color='#7C3AED')
    ax.set_title('Top 20 Feature Importance - meilleur modèle v6')
    savefig('v6_08_feature_importance.png')
else:
    print('Importance native indisponible pour ce modèle')

if RUN_SHAP:
    try:
        import shap
        X_sample = pd.DataFrame(BEST['preprocessor'].transform(test[features]) if BEST['preprocessor'] is not None else test[features], columns=features).sample(min(600, len(test)), random_state=RANDOM_STATE)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        shap.summary_plot(shap_values, X_sample, show=False)
        savefig('v6_09_shap_summary.png')
    except Exception as exc:
        print('SHAP non exécuté:', exc)


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_06_final_model_comparison.png


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_07_error_analysis.png


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\notebooks\figures\v6_08_feature_importance.png


## 12. Sauvegarde des artefacts v6

In [13]:
summary_path = ARTIFACTS_DIR / 'benchmark_phase1_v6_results.json'
model_path = ARTIFACTS_DIR / 'market_ranking_model_v6.joblib'

artifact = {
    'version': 'benchmark_phase1_v6_ranking_walkforward_optuna',
    'objective': 'direct ranking + walk-forward + business Optuna for Moroccan SME export market recommendation',
    'best_key': BEST_KEY,
    'best_model': str(summary.iloc[0]['model']),
    'best_target': str(summary.iloc[0]['target']),
    'best_feature_set': str(summary.iloc[0]['feature_set']),
    'features': BEST['features'],
    'categorical_features': CAT_FEATURES,
    'split': {
        'train_year_max': int(train['year'].max()),
        'val_years': [int(x) for x in val_years],
        'test_feature_year': int(latest_feature_year),
        'test_target_year': int(latest_feature_year + 1),
    },
    'audit': audit,
    'filter_funnel': funnel_df.to_dict('records'),
    'holdout_summary': summary.to_dict('records'),
    'walk_forward': wf_df.to_dict('records'),
    'optuna': OPTUNA_RESULTS,
    'figures': [
        'v6_01_data_quality_audit.png',
        'v6_02_filter_funnel.png',
        'v6_03_feature_diagnostics.png',
        'v6_04_target_comparison.png',
        'v6_05_walk_forward_best_model.png',
        'v6_06_final_model_comparison.png',
        'v6_07_error_analysis.png',
        'v6_08_feature_importance.png',
    ],
}

summary_path.write_text(json.dumps(artifact, indent=2, default=float), encoding='utf-8')
joblib.dump(BEST, model_path)
print('saved:', summary_path)
print('saved:', model_path)
print('BEST:', BEST_KEY)
display(summary.head(12))


saved: C:\Users\HP\Desktop\MaroTrade Intelligence\artifacts\benchmark_phase1_v6_results.json
saved: C:\Users\HP\Desktop\MaroTrade Intelligence\artifacts\market_ranking_model_v6.joblib
BEST: ExtraTrees__market_attractiveness__core_v5_1_stable


,key,model_type,model,target,feature_set,business_score,val_rmse,val_mae,val_r2,val_spearman,...,test_spearman,test_regret,test_top1,test_ndcg@1,test_top3,test_ndcg@3,test_top5,test_ndcg@5,test_groups,test_mean_regret
26,ExtraTrees__market_attractiveness__core_v5_1_s...,regression,ExtraTrees,market_attractiveness,core_v5_1_stable,0.350588,26.715146,20.998247,0.023129,0.163602,...,0.216342,19.167714,0.205882,0.770387,0.500000,0.809661,0.691176,0.832584,68,19.167714
32,ExtraTrees__market_attractiveness__core_plus_a...,regression,ExtraTrees,market_attractiveness,core_plus_advanced_v6,0.338097,26.801129,21.091652,0.016831,0.141506,...,0.201777,20.436810,0.176471,0.758328,0.485294,0.816499,0.647059,0.838947,68,20.436810
30,Ridge__market_attractiveness__core_plus_advanc...,regression,Ridge,market_attractiveness,core_plus_advanced_v6,0.334397,41.524302,22.011227,-1.360081,0.163050,...,0.221902,21.971694,0.220588,0.739772,0.573529,0.808972,0.676471,0.830923,68,21.971694
28,LightGBM__market_attractiveness__core_v5_1_stable,regression,LightGBM,market_attractiveness,core_v5_1_stable,0.334270,26.763626,21.043602,0.019580,0.188420,...,0.187297,21.152593,0.147059,0.746305,0.529412,0.801978,0.779412,0.838522,68,21.152593
45,LGBMRankerOptuna__market_attractiveness__core_...,ranker_optuna,LGBMRankerOptuna,market_attractiveness,core_plus_advanced_v6,0.331659,38.282171,32.692441,-1.005928,0.064887,...,-0.006881,24.316446,0.102941,0.710841,0.485294,0.773224,0.750000,0.811264,68,24.316446
31,Huber__market_attractiveness__core_plus_advanc...,regression,Huber,market_attractiveness,core_plus_advanced_v6,0.330949,43.035812,22.190907,-1.535024,0.144815,...,0.167355,19.598497,0.220588,0.764149,0.544118,0.806228,0.676471,0.832408,68,19.598497
34,LightGBM__market_attractiveness__core_plus_adv...,regression,LightGBM,market_attractiveness,core_plus_advanced_v6,0.326783,27.132785,21.354602,-0.007653,0.130831,...,0.183565,24.351088,0.147059,0.717618,0.588235,0.798121,0.735294,0.833113,68,24.351088
33,RandomForest__market_attractiveness__core_plus...,regression,RandomForest,market_attractiveness,core_plus_advanced_v6,0.325937,26.805864,21.112413,0.016483,0.148543,...,0.236964,20.737477,0.191176,0.750420,0.558824,0.811320,0.808824,0.847767,68,20.737477
29,XGBoost__market_attractiveness__core_v5_1_stable,regression,XGBoost,market_attractiveness,core_v5_1_stable,0.322870,26.802185,20.930309,0.016753,0.185076,...,0.191204,21.686069,0.147059,0.742528,0.558824,0.805361,0.750000,0.832621,68,21.686069
35,XGBoost__market_attractiveness__core_plus_adva...,regression,XGBoost,market_attractiveness,core_plus_advanced_v6,0.321765,27.235825,21.425212,-0.015321,0.133259,...,0.185200,24.521920,0.161765,0.713783,0.544118,0.795095,0.735294,0.830953,68,24.521920
